In [4]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys
import xarray as xr
from pathlib import Path
import importlib


# ============================================================
# HARD-CODED PROJECT PATHS
# Notebook location:
# THESIS/z.flow_postprocessing/notebooks/calculate_EDS_adjusted.ipynb
# ============================================================

NB_DIR = Path.cwd()

PROJECT_DIR = NB_DIR.parent.parent
OGCM_DIR = PROJECT_DIR / "OGCM"
OGCM_SCRIPTS_DIR = OGCM_DIR / "scripts"
THEME_DIR = PROJECT_DIR / "theme"

for path in [PROJECT_DIR, OGCM_SCRIPTS_DIR]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

print(f"Notebook dir     : {NB_DIR}")
print(f"Project dir      : {PROJECT_DIR}")
print(f"OGCM scripts dir : {OGCM_SCRIPTS_DIR}")
print(f"Theme dir        : {THEME_DIR}")


# ============================================================
# IMPORT PROJECT MODULES
# ============================================================

import theme.plot_theme as ptheme
import ocean_analysis as oa

importlib.reload(ptheme)
ptheme.apply_theme()

importlib.reload(oa)

Notebook dir     : c:\Users\Jelle Gortemaker\Documents\Thesis\z.flow_postprocessing\notebooks
Project dir      : c:\Users\Jelle Gortemaker\Documents\Thesis
OGCM scripts dir : c:\Users\Jelle Gortemaker\Documents\Thesis\OGCM\scripts
Theme dir        : c:\Users\Jelle Gortemaker\Documents\Thesis\theme


<module 'ocean_analysis' from 'c:\\Users\\Jelle Gortemaker\\Documents\\Thesis\\OGCM\\scripts\\ocean_analysis.py'>

In [5]:
# ============================================================
# CASE SETTINGS
# ============================================================

case_name = "run_feb1_basilisk_2D"
sim_name = "Case FEB-1 Basilisk 2D"
SAVE = False  

DATA_FILE = (NB_DIR / f"../data/input/{case_name}.nc").resolve()

# All figures are saved directly inside ../results/<case_name>/
RESULTS_DIR = (NB_DIR / f"../results/{case_name}").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save_case_figure(fig, figure_name, dpi=300, ext="png", close=False):
    """
    Save a figure inside ../results/<case_name>/.

    Example
    -------
    save_case_figure(fig, "EDS_overview_T3")
    """
    if not SAVE:
        return None

    figure_name = str(figure_name).replace(" ", "_")
    out_path = RESULTS_DIR / f"{figure_name}.{ext}"

    fig.savefig(
        out_path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white",
    )

    if close:
        plt.close(fig)

    print(f"Saved figure to: {out_path}")
    return out_path


In [6]:
# ============================================================
# SINGLE INITIALIZED-SIMULATION EDS OVERVIEW
# ============================================================

eds_init = oa.calculate_EDS_init(
    DATA_FILE,
    target_box=None,
    max_wavelength_km=None,
    x_res=419.252,
    y_res=435.859,
    n_bins=26,
    remove_mean=True,
    min_modes_per_bin=3,
    rose_scale_bands_km=[(110, 20)],
    rose_n_angle_bins=12,
    snapshot_index=None,
    layer_index=0,
)

if eds_init.attrs["snapshot_time_index"] is not None:
    # Open the file lazily to read the dimensions safely
    with xr.open_dataset(DATA_FILE) as ds:
        total_timesteps = ds.sizes['T']

    if total_timesteps >= 60:
        print(total_timesteps)
        n_days = eds_init.attrs["snapshot_time_index"] / 8
    else:
        print(total_timesteps)
        n_days = eds_init.attrs["snapshot_time_index"] / 2

    title = f"{sim_name} (T = {n_days:g} days)"
    time_label = f"T{eds_init.attrs['snapshot_time_index']}"
else:
    title = sim_name
    time_label = "all_timesteps"

fig = oa.plot_eds_overview(
    eds_init,
    title=title,
    ylim_a=(1e-8, 1e-2),      # subplot A y-axis range
    ylim_b=(1e-4, 1e4),       # subplot B y-axis range
    show=True,
)

save_case_figure(
    fig,
    f"{case_name}_{time_label}_EDS_overview",
)


<xarray.Dataset> Size: 732MB
Dimensions:    (time: 121, Z: 1, latitude: 514, longitude: 736)
Coordinates:
  * time       (time) datetime64[ns] 968B 2020-01-31 ... 2020-02-14T23:59:59....
  * Z          (Z) float32 4B 0.0
  * latitude   (latitude) float64 4kB 37.0 37.0 37.01 37.01 ... 39.0 39.01 39.01
    Y          (latitude) float64 4kB ...
  * longitude  (longitude) float64 6kB -147.8 -147.8 -147.7 ... -144.2 -144.2
    X          (longitude) float64 6kB ...
Data variables:
    UVEL       (time, Z, latitude, longitude) float64 366MB ...
    VVEL       (time, Z, latitude, longitude) float64 366MB ...
Attributes:
    domain:                   FEB-1 GLORYS box
    latitude_min:             37.0
    latitude_max:             40.0
    longitude_min:            -147.76
    longitude_max:            -144.24
    vertical_representation:  Single synthetic layer at Z=0 m representing a ...
Depth dimension: Z
Selected layer index: 0


ValueError: Dimensions {'Xp1'} do not exist. Expected one or more of FrozenMappingWarningOnValuesAccess({'time': 121, 'latitude': 514, 'longitude': 736})

In [ ]:
# # ============================================================
# # SEASONAL EDS PLOT
# # ============================================================
# # Edit paths/years here. This section uses GLORYS-style files through oa.calculate_EDS().

# seasonal_data_groups = {
#     "Winter": {
#         "2020": "../../OGCM/data/input/GPGP_jan2020.nc",
#         "2021": "../../OGCM/data/input/GPGP_jan2021.nc",
#         "2022": "../../OGCM/data/input/GPGP_jan2022.nc",
#     },
#     "Summer": {
#         "2020": "../../OGCM/data/input/GPGP_aug2020.nc",
#         "2021": "../../OGCM/data/input/GPGP_aug2021.nc",
#         "2022": "../../OGCM/data/input/GPGP_aug2022.nc",
#     },
# }

# # Target box for seasonal comparison.
# # Use None for whole domain, or e.g. [lon_w, lon_e, lat_s, lat_n].
# seasonal_target_box = None

# missing_files = []
# for season, years in seasonal_data_groups.items():
#     for year, rel_path in years.items():
#         candidate = (NB_DIR / rel_path).resolve()
#         if not candidate.exists():
#             missing_files.append(str(candidate))

# if missing_files:
#     print("Seasonal plot skipped because these files were not found:")
#     for path in missing_files:
#         print("  -", path)
# else:
#     fig_seasonal, ax_seasonal = oa.plot_EDS_seasonal(
#         data_groups=seasonal_data_groups,
#         target_box=seasonal_target_box,
#         initialized_velocity=False,
#         max_wavelength_km=None,
#         n_bins=40,
#         remove_mean=True,
#         temporal_window_days=1,
#         temporal_skip_days=0,
#         temporal_stride_days=1,
#         min_modes_per_bin=3,
#         spectrum_var="shell_integrated_spectrum",
#         title="Seasonal shell-integrated spectra comparison",
#         xlim_km=(20, 200),
#         ylim=None,  # set manually if desired, e.g. (1e-8, 1e-2)
#         show=True,
#     )

#     save_case_figure(
#         fig_seasonal,
#         f"{case_name}_seasonal_shell_integrated_spectra",
#     )
